we will learn about ShuffleHash,SortMerge,BroadCast join

---

## 📝 1. How Does a Join Work in Spark? (The Basic Idea)

**On a single computer, joining two tables is simple -- both
tables are in the same memory. But in Spark, the data is split
across multiple machines in a cluster. So how do matching rows
find each other?**

Let us start with two tables:

- **Sales** (Fact table) -- the big table
- **City** (Dimension table) -- the small table
- Both share a join key: `city_id`

```text
  Sales              City
  (Fact)             (Dimension)
  city_id            city_id
  --------           --------
     1                  1
     2                  2
     1                  3
     3                  4
     3                  5
     4
     1
     5
     2
```

Sales has 9 rows (many duplicates -- lots of sales in the same
city). City has only 5 rows (one per city). We want to join
them on `city_id`.

---

## 📝 2. After Reading -- Data Is Scattered Across Partitions

**When Spark reads Sales and City, it splits the rows into
partitions and sends each partition to a different executor in
the cluster. The data lands randomly -- there is no grouping
by `city_id`. Rows from both Sales and City are just scattered
everywhere.**

After reading, the 4 partitions might look like this:

```text
┌─────────────────────────────────────────┐
│               Cluster                        │
│                                               │
│  ┌─────────┐   ┌─────────┐               │
│  │   (1)    │   │   (2)    │               │
│  │         │   │         │               │
│  │   1     │   │   2  5  │               │
│  │   2     │   │         │               │
│  │   1     │   │         │               │
│  └─────────┘   └─────────┘               │
│                                               │
│  ┌─────────┐   ┌─────────┐               │
│  │   (3)    │   │   (4)    │               │
│  │         │   │         │               │
│  │   3  1  │   │   1  3  │               │
│  │   3  2  │   │   5  4  │               │
│  │   4     │   │         │               │
│  └─────────┘   └─────────┘               │
│                                               │
└─────────────────────────────────────────┘
```

Notice the mess: `city_id = 1` values are on Partition (1),
(3), and (4). `city_id = 3` is on (3) and (4). There is no
way to tell which number came from Sales and which came from
City. **Because the data is on different executors, we cannot
join -- an executor only sees its own partition.**

**To join, Spark must shuffle: move the data across the
network so that all rows with the same `city_id` end up on
the same partition.**

---

## 📝 3. After Shuffle -- Matching Keys Are Together

**Spark hashes each `city_id` and uses that hash to decide
which partition every row goes to. Same key = same hash =
same partition.**

After the shuffle, the cluster looks like this:

```text
┌─────────────────────────────────────────┐
│               Cluster                        │
│                                               │
│  ┌─────────────┐   ┌─────────────┐       │
│  │    (1)       │   │    (2)       │       │
│  │  S    C     │   │  S    C     │       │
│  │  --   --    │   │  --   --    │       │
│  │  1    1     │   │  2    2     │       │
│  │  1         │   │  2         │       │
│  │  1         │   │             │       │
│  └─────────────┘   └─────────────┘       │
│                                               │
│  ┌─────────────┐   ┌─────────────┐       │
│  │    (3)       │   │    (4)       │       │
│  │  S    C     │   │  S    C     │       │
│  │  --   --    │   │  --   --    │       │
│  │  3    3     │   │  5    5     │       │
│  │  3    4     │   │             │       │
│  │  4         │   │             │       │
│  └─────────────┘   └─────────────┘       │
│                                               │
└─────────────────────────────────────────┘
```

**Now every partition has matching `city_id` values from both
Sales (S) and City (C) side by side:**
1. **Partition (1):** All `city_id = 1` -- Sales has 3 rows,
   City has 1 row. They can join locally.
2. **Partition (2):** All `city_id = 2` -- Sales has 2 rows,
   City has 1 row.
3. **Partition (3):** `city_id = 3` and `city_id = 4` rows
   (the hash function grouped them here).
4. **Partition (4):** All `city_id = 5`.

**This is the fundamental idea behind every Spark join:** get
matching keys onto the same machine, then join locally. The
three strategies below differ in *how* they get the data
there and *how* they do the local matching.

---

## 📝 4. What Are Fact and Dimension Tables?

**Before we look at join strategies, a quick note on the
types of tables we typically join.**

In the example above, `Sales` is a **Fact table** -- it
stores the raw measurable events (the "facts" of your
business: what was sold, when, how much). Fact tables grow
continuously and can have billions of rows.

`City` is a **Dimension table** -- it gives descriptive
context ("dimensions") to the IDs in the fact table: city
name, state, population. Dimension tables are usually small
(hundreds to thousands of rows).

```text
┌──────────────────┐     ┌──────────────────┐
│ Sales (FACT)     │     │ City (DIMENSION)  │
├──────────────────┤     ├──────────────────┤
│ sale_id          │     │ city_id           │
│ city_id  ─────────────>│ city_name         │
│ amount           │     │ state             │
│ quantity         │     │ population        │
└──────────────────┘     └──────────────────┘
   millions of rows         hundreds of rows
```

**Why this matters:** The size difference between Sales (Fact)
and City (Dimension) is exactly what determines which join
strategy Spark picks.

| Strategy | When Spark uses it |
|---|---|
| **Broadcast Hash Join** | City is small enough to copy everywhere |
| **Shuffle Hash Join** | Both Sales and City are medium-sized |
| **Sort Merge Join** | Both Sales and City are large (default) |

---

In [1]:
# Spark Session
from pyspark.sql import SparkSession
spark = (
    SparkSession
    .builder
    .appName("Optimizing Joins")
    .master("spark://4619b568af8b:7077")
    .config("spark.cores.max", 16)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)
spark

In [2]:
# Disable AQE and Broadcast join
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [3]:
# Read EMP CSV data 10M records
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"
emp = spark.read.format("csv").schema(_schema).option("header", True).load("/data/csv/employee_records.csv")

In [4]:
# Read DEPT CSV data only 10 records
_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"
dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load("/data/csv/department_data.csv")

In [5]:
# Read Sales data 10M records
sales_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"
sales = spark.read.format("csv").schema(sales_schema).option("header", True).load("/data/csv/new_sales.csv")

In [12]:
# Read City data
city_schema = "city_id string, city string, state string, state_abv string, country string"
city = spark.read.format("csv").schema(city_schema).option("hearde",True).load("/data/csv/cities.csv")

In [8]:
# if one table is small and other is big one then we can use the broadcast join
from pyspark.sql.functions import broadcast
df_joined = emp.join(broadcast(dept), on=emp.department_id==dept.department_id, how="left_outer")
df_joined.write.format("noop").mode("overwrite").save()
df_joined.explain()

== Physical Plan ==
*(2) BroadcastHashJoin [department_id#7], [department_id#16], LeftOuter, BuildRight, false
:- FileScan csv [first_name#0,last_name#1,job_title#2,dob#3,email#4,phone#5,salary#6,department_id#7] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
+- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [id=#160]
   +- *(1) Filter isnotnull(department_id#16)
      +- FileScan csv [department_id#16,department_name#17,description#18,city#19,state#20,country#21] Batched: false, DataFilters: [isnotnull(department_id#16)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/department_data.csv], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<department_id:int,depa

## 📝 5. Broadcast Hash Join (BHJ) -- Copy City Everywhere

**If City is tiny (a few hundred rows), why shuffle the giant
Sales table at all? Just send City to every machine.**

That is exactly what a **Broadcast Join** does. Think of it
like photocopying a short reference sheet and handing one
copy to every student -- now everyone can look things up
locally without passing notes around.

```text
┌──────────────────────────────────────┐
│            Driver                        │
│  City (small) ---- broadcast --->        │
└──────┴────────┴────────┴────────────┘
       │        │        │
       ▼        ▼        ▼
  ┌───────┐ ┌───────┐ ┌───────┐
  │ Wkr 1 │ │ Wkr 2 │ │ Wkr 3 │
  │ Sales │ │ Sales │ │ Sales │
  │ part. │ │ part. │ │ part. │
  │   +   │ │   +   │ │   +   │
  │ City  │ │ City  │ │ City  │
  │(copy) │ │(copy) │ │(copy) │
  └───────┘ └───────┘ └───────┘
```

**How it works step by step:**
1. The Driver collects the entire City table into memory.
2. It broadcasts a full copy of City to every worker node.
3. Each worker builds a **hash table** from City in local
   memory.
4. Each worker scans its own Sales partition, looks up each
   Sales `city_id` in the City hash table, and when a match
   is found, the Sales row and City row are joined together.

**Key point:** Sales never moves. Zero shuffle on Sales =
fastest possible join.

**When to use it:** When City (or whichever table is smaller)
fits in memory (default threshold: 10 MB, configurable via
`spark.sql.autoBroadcastJoinThreshold`). max: 8Gb

---

## 📝 6. Shuffle Hash Join (SHJ) -- Hash and Shuffle

**What if both Sales and City are too large to broadcast?
Then Spark has to shuffle -- exactly what we saw in Sections
2 and 3.**

The difference from Sort Merge Join is what happens *after*
the shuffle: instead of sorting, Spark builds a hash table
from City (the smaller dataset) on each partition. Then it
takes each Sales row and checks it against the City hash
table -- when a Sales `city_id` matches a City `city_id`,
those two rows are joined together.

**How it works step by step:**
1. Spark hashes the `city_id` for every row in both Sales
   and City.
2. Rows with the same hash go to the same partition
   (**shuffle**) -- so all Sales and City rows with
   `city_id = 3` end up on the same machine.
3. On each partition, Spark builds a hash table from the
   City rows (because City is smaller, the hash table
   fits in memory).
4. Spark then scans each Sales row on that partition and
   looks up its `city_id` in the City hash table. When a
   match is found, those rows are joined.

**Downside:** Both Sales and City are shuffled across the
network -- that is expensive I/O.

---

In [7]:
# we are just doing a join nothing is specifically specified
df_joined = emp.join(dept, on=emp.department_id==dept.department_id, how="left_outer")
df_joined.write.format("noop").mode("overwrite").save()
df_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [department_id#7], [department_id#16], LeftOuter
:- *(1) Sort [department_id#7 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(department_id#7, 200), ENSURE_REQUIREMENTS, [id=#70]
:     +- FileScan csv [first_name#0,last_name#1,job_title#2,dob#3,email#4,phone#5,salary#6,department_id#7] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
+- *(3) Sort [department_id#16 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(department_id#16, 200), ENSURE_REQUIREMENTS, [id=#82]
      +- *(2) Filter isnotnull(department_id#16)
         +- FileScan csv [department_id#16,department_name#17,description#18,city#19,state#20,country#21] Batched: false, DataFilters: [isnotnull(department_id#16)], Format: CSV, Location: InMem

In [13]:
# now we are going 2 read two dataset with huge data in it 
"""new_sales 7M records and cities.csv with 2M records"""
df_sales_joined = sales.join(city, on=sales.city_id==city.city_id, how="left_outer")
df_sales_joined.write.format("noop").mode("overwrite").save()
df_sales_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [city_id#33], [city_id#170], LeftOuter
:- *(1) Sort [city_id#33 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(city_id#33, 200), ENSURE_REQUIREMENTS, [id=#240]
:     +- FileScan csv [transacted_at#28,trx_id#29,retailer_id#30,description#31,amount#32,city_id#33] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/new_sales.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit...
+- *(3) Sort [city_id#170 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(city_id#170, 200), ENSURE_REQUIREMENTS, [id=#252]
      +- *(2) Filter isnotnull(city_id#170)
         +- FileScan csv [city_id#170,city#171,state#172,state_abv#173,country#174] Batched: false, DataFilters: [isnotnull(city_id#170)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/cities.csv], PartitionFilters: [], P

## 📝 7. Sort Merge Join (SMJ) -- Shuffle, Sort, Walk Through

**This is Spark's default join for two large tables.** It
avoids building a hash table in memory (which could cause
OutOfMemory errors if City is not small enough) by sorting
the data first.

Think of two long alphabetically-sorted class lists. To find
students on both lists, you do not search the entire second
list for every name -- you walk down both together, advancing
whichever one is behind. That is a merge.

```text
Step 1: SHUFFLE     Step 2: SORT      Step 3: MERGE
(same as SHJ)      (within part.)   (walk both sides)

 Partition 1:       Partition 1:     Sales │ City
  S: 1, 1           S: 1, 1         ──────┼──────
  C: 1              C: 1               1  │  1  match
                                       1  │     match
                                       1  │     match

 Partition 2:       Partition 2:
  S: 3, 3, 2         S: 2, 3, 3       2  │  2  match
  C: 2, 3            C: 2, 3          3  │  3  match
                                      3  │     match
```

**Why does `1 |     match` work when the City column is empty?**
Because the merge uses two pointers -- one walking through Sales
rows and one walking through City rows. City has only 1 row with
`city_id = 1`, so the City pointer stays on that same row. But
Sales has 3 rows with `city_id = 1`, so the Sales pointer keeps
advancing. Each time it advances, the current Sales row still
matches the City row that the City pointer is sitting on. That
is why we get 3 joined output rows from just 1 City row -- the
City pointer does not move until the Sales pointer moves past
all the `city_id = 1` rows.

**How it works step by step:**
1. **Shuffle:** Both Sales and City are shuffled by
   `city_id`, so all Sales rows and City rows with the
   same `city_id` end up on the same partition.
2. **Sort:** Each partition sorts its Sales rows and City
   rows by `city_id`.
3. **Merge:** Two pointers walk through the sorted Sales
   and City data side by side. When a Sales `city_id`
   matches a City `city_id`, those rows are joined. If
   the next Sales row has the same `city_id`, the City
   pointer stays put and that Sales row also matches the
   same City row. Once the Sales `city_id` changes, the
   pointer on the smaller `city_id` advances.

**Why Spark defaults to this:** Sorting can spill to disk
when memory is tight (unlike hash tables that must fit in
RAM), so it works even when both Sales and City are
massive.

---

## 📝 8. Bucketing Strategy -- Pre-Sorting Data to Eliminate Shuffles

**In previous sections, every time we joined Sales and City, Spark had to shuffle millions of rows across the network. But what if we pre-group and store the data into buckets on disk ahead of time?**

This technique is called **Bucketing**.

---

### How Bucketing Works (MurmurHash & 4 Buckets)

Suppose we have:
* **Sales** with `city_id`: `1, 2, 1, 3, 4, 5, 1`
* **City** with `city_id`: `1, 2, 3, 4, 5`

We divide **both** tables into **4 buckets** on disk based on `city_id`.

When writing, Spark passes each `city_id` through **MurmurHash3**:
$$\text{bucket\_id} = \text{MurmurHash3}(\text{city\_id}) \pmod 4$$

```text
  TABLE 1: SALES (Writing 4 Buckets)
  city_id values: [ 1, 2, 1, 3, 4, 5, 1 ]
                      │
                      ▼
               ┌─────────────┐
               │ MurmurHash3 │
               └──────┬──────┘
       ┌──────────────┼──────────────┬──────────────┐
       ▼              ▼              ▼              ▼
  ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐
  │Bucket 1 │    │Bucket 2 │    │Bucket 3 │    │Bucket 4 │
  │ (Sales) │    │ (Sales) │    │ (Sales) │    │ (Sales) │
  ├─────────┤    ├─────────┤    ├─────────┤    ├─────────┤
  │ 1, 1, 1 │    │  3, 5   │    │    2    │    │    4    │
  └─────────┘    └─────────┘    └─────────┘    └─────────┘

  TABLE 2: CITY (Writing 4 Buckets)
  city_id values: [ 1, 2, 3, 4, 5 ]
                      │
                      ▼
               ┌─────────────┐
               │ MurmurHash3 │
               └──────┬──────┘
       ┌──────────────┼──────────────┬──────────────┐
       ▼              ▼              ▼              ▼
  ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐
  │Bucket 1 │    │Bucket 2 │    │Bucket 3 │    │Bucket 4 │
  │ (City)  │    │ (City)  │    │ (City)  │    │ (City)  │
  ├─────────┤    ├─────────┤    ├─────────┤    ├─────────┤
  │    1    │    │  3, 5   │    │    2    │    │    4    │
  └─────────┘    └─────────┘    └─────────┘    └─────────┘
```

---

### Why Does This Make Joins Instant (No Shuffle)?

When joining the two tables, Spark pairs up matching bucket numbers **1-to-1**:

```text
  ┌────────────────────────────────────────────────────────┐
  │                 JOINING BUCKET-TO-BUCKET               │
  │                                                        │
  │  Sales Bucket 1  ────── matches ──────>  City Bucket 1 │
  │  Sales Bucket 2  ────── matches ──────>  City Bucket 2 │
  │  Sales Bucket 3  ────── matches ──────>  City Bucket 3 │
  │  Sales Bucket 4  ────── matches ──────>  City Bucket 4 │
  └────────────────────────────────────────────────────────┘
```

1. **Zero Shuffle at Query Time:** Executor 1 reads `Sales Bucket 1` + `City Bucket 1`, Executor 2 reads `Sales Bucket 2` + `City Bucket 2`, and so on.
2. **Matching rows are already co-located:** Since MurmurHash is deterministic, `city_id = 1` will **never** be in Bucket 2, 3, or 4. Spark merges the corresponding buckets locally without sending any data over the network!

---

### Why Must We Use `saveAsTable()` Instead of Plain File Save?

* **Files on disk do not carry bucketing metadata:** If you only use `.save("/path")`, Spark writes raw CSV files, but does not record that the data inside is bucketed by `city_id` into 4 buckets. When you read it with `spark.read.csv()`, Spark treats it as random data and **still shuffles**.
* **`saveAsTable("sales_bucket")` registers permanent metadata in the Spark Catalog:**
  1. It permanently records that the table is bucketed into **4 buckets on `city_id`**.
  2. It links the table name `sales_bucket` to the physical disk folder (`/data/csv/sales_bucket.csv`).
* When you query via `spark.table("sales_bucket")`, Spark checks the catalog, detects the bucketing metadata, and **skips the Shuffle step** automatically.

#### 💡 Is `saveAsTable()` In-Memory (Temporary) or Permanent?
**It is 100% Permanent (Persistent on Disk)!**

| Feature | `createOrReplaceTempView("view_name")` | `saveAsTable("table_name")` |
| :--- | :--- | :--- |
| **Persistence** | **Temporary (Session-only)** | **Permanent (Persistent)** |
| **Where is it stored?** | In memory for active session only | **On Disk** (data files + catalog metadata) |
| **After session restart?** | **Gone.** Must recreate. | **Still there!** Available tomorrow via `spark.table()`. |
| **Supports Bucketing?** | ❌ No | ✅ Yes |

---

### Understanding the Saved Output Files

When you open the output folder `/data/csv/sales_bucket.csv/`, you will see files named like:
`part-00000-..._00002.c000.csv`

Here is what the parts of the filename mean:
* `part-00000`: The **Write Task / Partition ID** that wrote this file.
* `_00002`: The **Bucket ID** (ranges from `_00000` to `_00003` for 4 buckets).
* If your writing stage had 4 partitions and each wrote into all 4 buckets, you will see up to $4 \times 4 = 16$ files (numbered from task 0..3 and bucket 0..3).

---

In [ ]:
# path option("path","/data/csv/sales_bucket.csv") is the output path
sales.write.format("csv").mode("overwrite").bucketBy(4,"city_id").option("header",True).option("path","/data/csv/sales_bucket.csv").saveAsTable("sales_bucket")

In [16]:
# similarly we write for the city
city.write.format("csv").mode("overwrite").bucketBy(4,"city_id").option("header",True).option("path","/data/csv/city_bucket.csv").saveAsTable("city_bucket")

In [17]:
spark.sql("show tables in default").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default| city_bucket|      false|
|  default|sales_bucket|      false|
+---------+------------+-----------+



#### ☝️ What just happened -- checking the catalog tables

* **`spark.sql("SHOW TABLES IN default")`**: Runs a SQL command directly through Spark's SQL engine to list all registered tables in the built-in `default` database.
* **`tableName`**: Confirms that our bucketed `sales_bucket` and `city_bucket` tables exist in the catalog.
* **`isTemporary = false`**: Proves these are **permanent/persistent tables** (not temporary views). Their schema, storage path, and 4-bucket metadata remain in the catalog even after restarting your session.

---

In [19]:
# Read Sales table
sales_bucket = spark.read.table("sales_bucket")
# Read City table
city_bucket = spark.read.table("city_bucket")

In [20]:
# Join datasets
df_joined_bucket = sales_bucket.join(city_bucket, on=sales_bucket.city_id==city_bucket.city_id, how="left_outer")
# Write dataset
df_joined_bucket.write.format("noop").mode("overwrite").save()

In [21]:
df_joined_bucket.explain()

== Physical Plan ==
*(3) SortMergeJoin [city_id#456], [city_id#463], LeftOuter
:- *(1) Sort [city_id#456 ASC NULLS FIRST], false, 0
:  +- FileScan csv default.sales_bucket[transacted_at#451,trx_id#452,retailer_id#453,description#454,amount#455,city_id#456] Batched: false, Bucketed: true, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/sales_bucket.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit..., SelectedBucketsCount: 4 out of 4
+- *(2) Sort [city_id#463 ASC NULLS FIRST], false, 0
   +- *(2) Filter isnotnull(city_id#463)
      +- FileScan csv default.city_bucket[city_id#463,city#464,state#465,state_abv#466,country#467] Batched: false, Bucketed: true, DataFilters: [isnotnull(city_id#463)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/csv/city_bucket.csv], PartitionFilters: [], PushedFilters: [IsNotNull(city_id)], ReadSchema: str

## 📝 9. Key Rules and Best Practices for Bucketed Joins

**In the physical plan above, notice there is NO `Exchange` (Shuffle) step! Spark scanned the files with `Bucketed: true` and merged them directly without transferring data across the network.**

Here is how Spark behaves across different bucketing scenarios:

---

### 📌 Behavior Matrix for Bucketed Joins

| # | Join Column vs Bucket Column | Bucket Count | Resulting Shuffle Behavior |
|---|---|---|---|
| **1** | **Different column** than bucket column | Same bucket size | **Shuffle on BOTH tables** (bucketing is bypassed) |
| **2** | **Same join column**, but only 1 table is bucketed | One bucketed, one regular | **Shuffle on the non-bucketed table** |
| **3** | **Same join column**, but different bucket sizes | Different bucket counts | **Shuffle on the smaller bucket side** (to repartition to match) |
| **4** | **Same join column AND same bucket size** | Identical (e.g., 4 and 4) | **NO SHUFFLE! (Fastest Join)** 🚀 |

---

### 💡 Best Practices & Practical Tips

1. **Choose the correct bucket column:** Always bucket on the columns that are frequently used in **`JOIN` conditions or `GROUP BY` clauses** (e.g., `city_id`, `customer_id`).
2. **Decide effectively on the number of buckets:** 
   * Having **too many buckets** with small data creates the **Small File Problem** (thousands of tiny files on disk), which degrades query planning and I/O performance.
   * Target bucket file sizes around 128 MB -- 256 MB.
3. **When datasets are small:** If the dimension table is small enough to fit in memory, prefer **Broadcast Hash Join** over bucketing to avoid the upfront write overhead of pre-bucketing.

---